Set up the environment

In [ ]:
# install ultralytics for Yolo
!pip install ultralytics
!pip install boxmot==17.0.0

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import os
from ultralytics import YOLO
from boxmot import create_tracker
from pathlib import Path
import yaml
from collections import defaultdict
import cv2
import csv
import time

In [ ]:
# configure the data path
DRIVE_ROOT = '/content/drive/MyDrive/16th grade Academics/A S26/CSCI 6980'
LOCAL_WORK_PATH = 'content/hockey_processing_folder'
YOLO_PATH = os.path.join(DRIVE_ROOT, 'YOLO_Results')
YAML_PATH = os.path.join(DRIVE_ROOT, 'YOLO_HOCKEY_DATASET/hockey_data.yaml')
VIDEO_PATH = os.path.join(DRIVE_ROOT, 'videos')
CSV_PATH = os.path.join(DRIVE_ROOT, 'player_locations')

Tracking

In [ ]:
MODEL_PATH = os.path.join(YOLO_PATH, 'hockey_model_v2/weights/best.pt')
best_model = YOLO(MODEL_PATH)

# light
reid_model = 'osnet_x0_25_msmt17.pt'
# medium
# reid_model = 'osnet_x1_0_msmt17.pt'
# # heavy
# reid_model = 'resnet50_msmt17.pt'

reid_weights = Path(reid_model)

In [ ]:
# allow user to input video name
video_name = 'tom_annotated'
input_video_path = os.path.join(VIDEO_PATH, f"{video_name}.mov")
output_video_path = os.path.join(VIDEO_PATH, f"{video_name}_StrongSORT_{reid_model}.mp4")
output_csv_path = os.path.join(CSV_PATH, f"{video_name}_StrongSORT_{reid_model}.csv")

In [ ]:
from boxmot import create_tracker
from pathlib import Path
import yaml

# strongsort specific config
strongsort_cfg = {
    'max_dist': {'default': 0.3},
    'max_iou_dist': {'default': 0.7},
    'max_age': {'default': 30},
    'n_init': {'default': 1},
    'nn_budget': {'default': 100}
}

# save the yaml
config_path = 'fixed_strongsort_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(strongsort_cfg, f)

# initialize the tracker
tracker = create_tracker(
    tracker_type='strongsort',
    tracker_config=config_path,
    reid_weights=reid_weights,
    device='cuda:0',
    half=True
)

SUCCESS  | StrongSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False, asso_func=iou, min_conf=0.1, max_cos_dist=0.2, max_iou_dist=0.7, n_init=1, nn_budget=100, mc_lambda=0.98, ema_alpha=0.9, max_dist=0.3
INFO     | BoxMOT v16.0.11 🚀 Python-3.12.13 torch-2.10.0+cu128
CUDA:0 (Tesla T4, 14913MiB)
INFO     | /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt
INFO     | [PID 2533] Downloading ReID weights from https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF → /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt
Downloading...
From: https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF
To: /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt
100%|██████████| 3.06M/3.06M [00:00<00:00, 245MB/s]
SUCCESS  | Loaded pretrained weights from /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt


In [ ]:
track_history = defaultdict(lambda: [])

cap = cv2.VideoCapture(input_video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))
out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

In [ ]:
# generate a consistent, unique color based on the ID
def get_color(track_id):
    b = (int(track_id) * 37) % 255
    g = (int(track_id) * 71) % 255
    r = (int(track_id) * 131) % 255
    return (b, g, r)

In [ ]:
export_data = []
frame_number = 1

print("Starting processing")
start_time = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # fetch YOLO detections for Goalie, Player, and Ref
    results = best_model.predict(frame, conf=0.33, iou=0.33, device=0, classes=[3, 4, 6], agnostic_nms = True, verbose=False)

    # extract the detection's [x1, y1, x2, y2, conf, class_id]
    det = results[0].boxes.data.cpu().numpy()

    # feed the tracker the new frames and boxes
    tracks = tracker.update(det, frame)

    # tracks output format: [x1, y1, x2, y2, track_id, class_id, conf]
    if tracks.shape[0] > 0:
        for t in tracks:
            x1, y1, x2, y2, track_id, class_id, conf = t[:7]

            # retrieve unqiue color for the id
            player_color = get_color(track_id)

            # using middle middle of player to match gt
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2

            # append data for csv
            export_data.append([frame_number, int(track_id), float(cx), float(cy)])

            # append to history
            track = track_history[int(track_id)]
            track.append((float(cx), float(cy)))
            if len(track) > 60:
                track.pop(0)

            # draw path in video
            if len(track) > 1:
                points = np.array(track).astype(np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [points], isClosed=False, color=player_color, thickness=2)

            # draw bb and ID in video
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), player_color, 2)
            cv2.putText(frame, f"ID:{int(track_id)}", (int(x1), int(y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, player_color, 2)

    out.write(frame)
    frame_number += 1

end_time = time.time()
total_time = end_time - start_time
total_processed_frames = frame_number - 1
average_fps = total_processed_frames / total_time

cap.release()
out.release()
print(f"Complete! Saved to: {output_video_path}")

print(f"--- PERFORMANCE METRICS ---")
print(f"Total Frames: {total_processed_frames}")
print(f"Total Time: {total_time:.2f} seconds")
print(f"Processing Speed: {average_fps:.2f} FPS")

Starting processing
Complete! Saved to: /content/drive/MyDrive/16th grade Academics/A S26/CSCI 6980/videos/tom_annotated_StrongSORT_osnet_x0_25_msmt17.pt.mp4
--- PERFORMANCE METRICS ---
Total Frames: 402
Total Time: 87.76 seconds
Processing Speed: 4.58 FPS


In [ ]:
# write data to the csv
with open(output_csv_path, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Frame', 'ID', 'X_Coordinate', 'Y_Coordinate'])
    writer.writerows(export_data)

print(f"Complete! Saved to: {output_csv_path}")

Complete! Saved to: /content/drive/MyDrive/16th grade Academics/A S26/CSCI 6980/player_locations/tom_annotated_StrongSORT_osnet_x0_25_msmt17.pt.csv
